In [1]:
import pandas as pd
import numpy as np
from scipy.constants import pi

branch_data = pd.read_csv("branch.csv")
h_7 = 7 * 50

def create_z_7(branch_data):
    return [
        complex(row['r'], row['l'] * 2 * pi * h_7)
        for _, row in branch_data.iterrows()
    ]

def create_y_7(branch_data):
    return [
        complex(0, row['c'] * 2 * pi * h_7)
        for _, row in branch_data.iterrows()
    ]

def create_gammaL_7(branch_data, z_7, y_7):
    z_7 = np.array(z_7)
    y_7 = np.array(y_7)
    gammaL_7 = []
    for i, row in branch_data.iterrows():
        length = row['Length']
        gammaL = length * np.sqrt(z_7[i] * y_7[i])
        gammaL_7.append(gammaL)
    return gammaL_7

z_7 = create_z_7(branch_data)
y_7 = create_y_7(branch_data)
gammaL_7 = create_gammaL_7(branch_data, z_7, y_7)

def Z0_7(z_7, y_7):
    return np.sqrt(np.array(z_7) / np.array(y_7))

Z0_7 = Z0_7(z_7, y_7)
def ABCD_7(gammaL_7, Z0_7):
    A = np.cosh(gammaL_7)
    B = Z0_7 * np.sinh(gammaL_7)
    C = (1 / Z0_7) * np.sinh(gammaL_7)
    D = np.cosh(gammaL_7)
    return A, B, C, D
A_7, B_7, C_7, D_7 = ABCD_7(gammaL_7, Z0_7)
ABCD_7_df = pd.DataFrame({
    'From Bus Number': branch_data['From Bus  Number'],
    'To Bus Number'  : branch_data['To Bus  Number'],
    'A_7'           : A_7,
    'B_7'           : B_7,
    'C_7'           : C_7,
    'D_7'           : D_7
})

ABCD_7_df.to_csv("ABCD_7.csv", index=False)


In [2]:
import pandas as pd
import numpy as np

# Load and ensure numeric types
ABCD = pd.read_csv("ABCD_7.csv")

# Normalize column names in case the CSV contains extra whitespace
ABCD.columns = ABCD.columns.str.strip()

# Convert to complex numbers safely
for col in ['A_7', 'B_7', 'C_7', 'D_7']:
    ABCD[col] = ABCD[col].apply(lambda x: complex(x.replace('i', 'j')) if isinstance(x, str) else complex(x))

def combine_parallel(group):
    from_bus, to_bus = group.name

    if len(group) == 1:
        row = group.iloc[0]
        return pd.Series({
            'From Bus Number': from_bus,
            'To Bus Number': to_bus,
            'A': row['A_7'],
            'B': row['B_7'],
            'C': row['C_7'],
            'D': row['D_7']
        })

    # Start with first line
    A_eq, B_eq, C_eq, D_eq = group.iloc[0][['A_7', 'B_7', 'C_7', 'D_7']]
    for _, row in group.iloc[1:].iterrows():
        A1, B1, C1, D1 = A_eq, B_eq, C_eq, D_eq
        A2, B2, C2, D2 = row['A_7'], row['B_7'], row['C_7'], row['D_7']

        A_eq = (A1 * B2 + A2 * B1) / (B1 + B2)
        B_eq = (B1 * B2) / (B1 + B2)
        C_eq = C1 + C2 + ((A1 - A2) * (D1 - D2)) / (B1 + B2)
        D_eq = (D1 * B2 + D2 * B1) / (B1 + B2)

    return pd.Series({
        'From Bus Number': from_bus,
        'To Bus Number': to_bus,
        'A': A_eq, 'B': B_eq, 'C': C_eq, 'D': D_eq
    })

# Combine lines with same (From, To)
ABCD_combined = (
    ABCD.groupby(['From Bus Number', 'To Bus Number'])
    .apply(combine_parallel)
    .reset_index(drop=True)
)

# Save result
ABCD_combined.to_csv("ABCD_parallel_combined_7.csv", index=False)

print(ABCD_combined.shape)


(27, 6)


C:\Users\User\AppData\Local\Temp\ipykernel_4724\2134324236.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(combine_parallel)


In [3]:
import pandas as pd
import numpy as np

# Load data
ABCD_combined = pd.read_csv("ABCD_parallel_combined_7.csv")

def abcd2s(df):
    S_data = []

    Z0 = 360.16
    Y0 = 1 / Z0

    for _, row in df.iterrows():
        from_bus = row["From Bus Number"]
        to_bus   = row["To Bus Number"]

        A = complex(row["A"])
        B = complex(row["B"])
        C = complex(row["C"])
        D = complex(row["D"])

        denom = (A + B*Y0 + C*Z0 + D)

        S11 = (A + B*Y0 - C*Z0 - D) / denom
        S12 = 2 * (A*D - B*C) / denom
        S21 = 2 / denom
        S22 = (-A + B*Y0 - C*Z0 + D) / denom

        S = np.array([
            [S11, S12],
            [S21, S22]
        ], dtype=complex)

        S_data.append({
            "from_bus": from_bus,
            "to_bus": to_bus,
            "S": S
        })

    return S_data
S_params = abcd2s(ABCD_combined)

for i, item in enumerate(S_params):
    print(f"\nLine {i}: Bus {item['from_bus']} → Bus {item['to_bus']}")
    print(item["S"])



Line 0: Bus (2135+0j) → Bus (2220+0j)
[[-0.04552579-0.1787695j   0.95142935-0.24468247j]
 [ 0.95142935-0.24468247j -0.04552579-0.1787695j ]]

Line 1: Bus (2135+0j) → Bus (2400+0j)
[[-0.68401024-0.1981089j   0.18797721-0.66779942j]
 [ 0.18797721-0.66779942j -0.68401024-0.1981089j ]]

Line 2: Bus (2135+0j) → Bus (2970+0j)
[[-0.20919486-0.33553851j  0.77634748-0.4878359j ]
 [ 0.77634748-0.4878359j  -0.20919486-0.33553851j]]

Line 3: Bus (2220+0j) → Bus (2225+0j)
[[-0.01689769-0.10207202j  0.97872693-0.1715329j ]
 [ 0.97872693-0.1715329j  -0.01689769-0.10207202j]]

Line 4: Bus (2220+0j) → Bus (2230+0j)
[[-0.08361289-0.23615297j  0.91076325-0.32574628j]
 [ 0.91076325-0.32574628j -0.08361289-0.23615297j]]

Line 5: Bus (2220+0j) → Bus (2570+0j)
[[-0.33778638-0.37120148j  0.63557828-0.58296852j]
 [ 0.63557828-0.58296852j -0.33778638-0.37120148j]]

Line 6: Bus (2220+0j) → Bus (2691+0j)
[[-0.39626924-0.28403792j  0.49466787-0.70926675j]
 [ 0.49466787-0.70926675j -0.39626924-0.28403792j]]

Line 

In [7]:
import pandas as pd
import numpy as np
from scipy.signal import find_peaks

# 1. Load the branch data
# If the script still pulls in junk data, try changing to: pd.read_csv('branch.csv', header=2)
df = pd.read_csv('branch.csv')

# Clean headers: Strip outer whitespace and replace multiple inner spaces with a single space
df.columns = df.columns.str.strip().str.replace(r'\s+', ' ', regex=True)

# Debug print to verify exactly what pandas is seeing
print("Detected Columns:", df.columns.tolist())

# Base parameters
f0 = 50.0  
max_harmonic = 50  
resolution = 0.1   

# 2. Extract unique buses using the cleaned, single-space column names
buses = pd.concat([df['From Bus Number'], df['To Bus Number']]).dropna().unique()
num_buses = len(buses)
bus_to_idx = {bus: i for i, bus in enumerate(buses)}
idx_to_bus = {i: bus for i, bus in enumerate(buses)}
Z_ii_mag = {bus: [] for bus in buses}
harmonics = np.arange(1, max_harmonic + resolution, resolution)

print("\nCalculating frequency-dependent Z-bus matrices. This may take a moment...")

# 3. Frequency Sweep
for h in harmonics:
    Y_bus = np.zeros((num_buses, num_buses), dtype=complex)
    
    for _, row in df.iterrows():
        # Update these two lines inside your loop:
        from_bus = row['From Bus Number']
        to_bus = row['To Bus Number']
        
        # Skip invalid rows
        if pd.isna(from_bus) or pd.isna(to_bus):
            continue
            
        from_idx = bus_to_idx[from_bus]
        to_idx = bus_to_idx[to_bus]
        
        # Fundamental per-unit parameters
        R = row['Line R (pu)']
        X = row['Line X (pu)']
        B = row['Charging B (pu)']
        
        # Frequency-dependent branch parameters
        Y_series = 1.0 / (R + 1j * h * X)
        Y_shunt = 1j * h * (B / 2.0)
        
        # Build Y-bus
        Y_bus[from_idx, from_idx] += Y_series + Y_shunt
        Y_bus[to_idx, to_idx] += Y_series + Y_shunt
        Y_bus[from_idx, to_idx] -= Y_series
        Y_bus[to_idx, from_idx] -= Y_series

    # Add a microscopic shunt admittance to avoid singular matrices if the network is floating
    np.fill_diagonal(Y_bus, Y_bus.diagonal() + 1e-8)
    
    # Invert Y-bus to get Z-bus
    Z_bus = np.linalg.inv(Y_bus)
    
    # Store the driving-point impedance magnitude for each bus
    for i in range(num_buses):
        Z_ii_mag[idx_to_bus[i]].append(np.abs(Z_bus[i, i]))

# 4. Identify and Output Resonance Frequencies
print("\n--- Network Resonance Frequencies by Substation ---")
for bus in buses:
    z_mag = np.array(Z_ii_mag[bus])
    
    # Find local maxima in the impedance magnitude curve
    # prominence threshold filters out minor ripples
    peaks, _ = find_peaks(z_mag, prominence=0.5) 
    
    res_freqs = harmonics[peaks] * f0
    
    if len(res_freqs) > 0:
        formatted_freqs = ", ".join([f"{freq:.1f} Hz" for freq in res_freqs])
        
        # FIXED: Using single spaces to match the cleaned headers
        bus_name_row = df[df['From Bus Number'] == bus]
        bus_name = bus_name_row['From Bus Name'].iloc[0].strip() if not bus_name_row.empty else f"Bus {bus}"
        
        print(f"{bus_name} (ID: {int(bus)}): {formatted_freqs}")

Detected Columns: ['From Bus Number', 'From Bus Name', 'To Bus Number', 'To Bus Name', 'Id', 'Line', 'Line R (pu)', 'Line X (pu)', 'Charging B (pu)', 'Length', 'r', 'x', 'b', 'l', 'c', 'z', 'y', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23']

Calculating frequency-dependent Z-bus matrices. This may take a moment...

--- Network Resonance Frequencies by Substation ---
NPOLP_2     220.00 (ID: 2135): 280.0 Hz, 525.0 Hz, 720.0 Hz, 1245.0 Hz, 1600.0 Hz, 1870.0 Hz
KOTMA-2     220.00 (ID: 2220): 280.0 Hz, 525.0 Hz, 1025.0 Hz, 1115.0 Hz, 1250.0 Hz, 1870.0 Hz
BARGE-2     220.00 (ID: 2222): 390.0 Hz, 1025.0 Hz, 1115.0 Hz, 1245.0 Hz, 1600.0 Hz, 1775.0 Hz, 1870.0 Hz, 2160.0 Hz
VICTO-2     220.00 (ID: 2230): 280.0 Hz, 525.0 Hz, 1025.0 Hz, 1115.0 Hz, 1245.0 Hz, 1600.0 Hz, 1775.0 Hz, 1870.0 Hz
RANDE-2     220.00 (ID: 2240): 280.0 Hz, 525.0 Hz, 1025.0 Hz, 1115.0 Hz, 1245.0 Hz, 1600.0 Hz, 1775.0 Hz, 1870.0 Hz
VAVUNIYA-2  220.00 (ID: 2245): 280.0